# Prepare Excel file for sequence submission 
Krista Longnecker, 3 September 2026

Have an Excel file that will go to Dalhouise, need to make a sheet for BIOS-SCOPE-specific details. The Excel file template is from Dalhousie's IMR group

In [114]:
%reset -f

In [115]:
import os
import re
import pandas as pd
import xlwings as xw #need to use this to make the new sheet to keep the Dalhousie validations
import pdb #use with set_trace()

In [116]:
data_dir = 'seq_data/'
fName = 'Submissions/IMR-SubmissionTemplate_V4_4_BIOSSCOPE 2026 Sept.xlsx'
sheet = 'Amplicon - MiSeq'

In [117]:
# make my NewID (or a placeholder if I don't have cast and Niskin information)
def makeID(possible,row):
    #pdb.set_trace()
    #do we have the data we need from BATS to determine cast and Niskin information?
    if possible == 1:
        one = row['cruise5'] + f"{row['cast']:02d}" + f"{row['niskin']:02d}"
    else:
        one = row['cruise5'] + 'ccc' + 'nn'
    
    return one

In [118]:
#parsing out the information in the filenames, use a function, this works with the NCBI filenames
def parseSample(sample):
    #parsed = {} #use the next row and setup the dictionary first
    #even empty it is useful as I don't have to repeat all the NA options
    parsed = dict({'cruise5':'',
                   'cast':'',
                   'niskin':'',
                   'cruise':'',
                   'otherInfo':'',
                   'depth':'',
                   'sampleNumber':'',
                   'New_Bottle_ID':'',
                   'seqType':''})
    #start with 10 digits - cruise - cast - niskin and then underscores to note 18S_V4
    ru = [match.start() for match in re.finditer("_",sample)]
    #print(sample)
    if len(ru) ==2 & ru[0]==10:
        #pdb.set_trace()
        parsed['cruise5'] = sample[:5]
        parsed['cast'] = sample[5:8]
        parsed['niskin']= sample[8:10]        
        parsed['New_Bottle_ID'] = sample[:10]
        parsed['otherInfo'] = sample[ru[0]+1:]
        if '18S' in parsed['otherInfo']:
            parsed['seqType'] = 'V4_18s'
        elif '16S_V1V2' in parsed['otherInfo']:
            parsed['seqType'] = 'V1V2'
    elif (len(ru) ==2) & (ru[0] ==5):
        parsed['cruise5'] = sample[:5]
        parsed['sampleNumber'] = sample[ru[0]+1 : ru[1]]
        parsed['depth'] = sample[ru[1]+1 : -1]
            
    parsed = pd.DataFrame([parsed]) 
    return parsed

In [119]:
# box with names is B31 to M38
df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName),sheet_name = sheet,usecols = 'B:M',skiprows = 30, nrows = 8,header = None,dtype=str))
df

,1,2,3,4,5,6,7,8,9,10,11,12
0,NaN,NaN,NaN,NaN,NaN,NaN,10414_1_0m,10414_9_500m,10415_5_160m,20415_1_0m,10416_1_0m,10416_9_500m
1,NaN,NaN,NaN,NaN,NaN,NaN,10414_2_40m,10414_10_600m,10415_6_200m,20415_2_40m,10416_2_40m,10416_10_600m
2,NaN,NaN,NaN,NaN,NaN,NaN,10414_3_80m,10414_11_800m,10415_7_250m,20415_3_80m,10416_3_80m,10416_11_800m
3,NaN,NaN,NaN,NaN,NaN,NaN,10414_4_120m,10414_12_1000m,10415_8_300m,20415_4_120m,10416_4_120m,10416_12_1000m
4,NaN,NaN,NaN,NaN,NaN,NaN,10414_5_160m,10415_1_0m,10415_9_500m,20415_5_160m,10416_5_160m,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,10414_6_200m,10415_2_40m,10415_10_600m,20415_6_200m,10416_6_200m,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,10414_7_250m,10415_3_80m,10415_11_800m,20415_7_250m,10416_7_250m,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,10414_8_300m,10415_4_120m,10415_12_1000m,20415_8_300m,10416_8_300m,NaN


In [120]:
#make one column and put sampleName for the first column's header
df = df.stack().reset_index(drop = True).to_frame()
df = df.rename(columns= {0: 'sampleName'})

In [121]:
df.head()

,sampleName
0,10414_1_0m
1,10414_9_500m
2,10415_5_160m
3,20415_1_0m
4,10416_1_0m


In [122]:
for idx,row in df.iterrows():
    #pdb.set_trace()
    sample = row['sampleName']
    parsed = parseSample(sample)
    df.at[idx,'cruise5'] = parsed.loc[0,'cruise5']
    df.at[idx,'sampleNumber'] = parsed.loc[0,'sampleNumber']
    df.at[idx,'depth'] = parsed.loc[0,'depth']

    del(sample,parsed)  

#make depth and sampleNumber both a number
df['depth'] = df['depth'].astype('Int64')
df['sampleNumber'] = df['sampleNumber'].astype('Int64')
df = df.sort_values(by = ['cruise5','sampleNumber'])

df['NewID'] = None

In [123]:
for idx,row in df.iterrows():
    possible = 0 #do NOT have what we need from BATS
    df.at[idx,'NewID'] = makeID(0,row) #makeID is the function defined above   

In [124]:
df.head()

,sampleName,cruise5,sampleNumber,depth,NewID
0,10414_1_0m,10414,1,0,10414cccnn
6,10414_2_40m,10414,2,40,10414cccnn
12,10414_3_80m,10414,3,80,10414cccnn
18,10414_4_120m,10414,4,120,10414cccnn
24,10414_5_160m,10414,5,160,10414cccnn


In [125]:
#add the columns to be filled in later:
toFill = 'fill in when sequencing is complete'
df['gzipFilename_1'] = None
df['gzipFilename_2'] = None
df['gzipFilename_3'] = None
df['seqLocation'] = None

df.loc[0,['gzipFilename_1','gzipFilename_2','gzipFilename_3','seqStorageLocation']] = toFill

In [126]:
# Use xlwings to put the new sheet in the Excel file
with xw.App(visible=False) as app:
    wb = xw.Book(os.path.join(data_dir,fName))
    
    #add the BIOS-SCOPE details to the worksheet
    new_ws = wb.sheets.add(name="projectDetails", after=wb.sheets[-1])
    new_ws.range("A1").options(index=False).value = df
    
    wb.save()
    wb.close()

Below here will not run and is a holding zone for misc. code bits

In [127]:
#Stick some code below this spot as a holding zone
raise SystemExit("Stop execution here")

SystemExit: Stop execution here

C:\Users\LaneyLab\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3465: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
dfm = df.merge(bn,on = 'sampleNumber',how = 'left')
dfm = dfm.sort_values(by = ['cruise5','depth_x']).reset_index(drop= True) #for now keep both depth columns
dfm

In [ ]:
#do NOT do this...it does vary by cruise so I don't want to guess




## now, how to match to sample numbers...these are fixed for BIOS-SCOPE casts (still do not have access to the BATS cast sheets)
#make a lookup table:
bn = pd.DataFrame({
    'sampleNumber':range(1,13,1),
    'niskin': [1,3,5,8,10,11,13,15,17,19,21,23],
    'depth':[1,40,80,120,160,200,250,300,500,600,800,1000]})

bn['sampleNumber'] = bn['sampleNumber'].astype('Int64')
df['sampleNumber'] = df['sampleNumber'].astype('Int64')
 

In [ ]:
import openpyxl

# 1. Load the workbook and keep VBA macros
file_path = os.path.join(data_dir,'Submissions/testing.xlsx')
wb = openpyxl.load_workbook(file_path, keep_vba=True)

# 2. Add a new worksheet (specify name and optional position index)
# index=0 forces it to be the first sheet; omit index to append to the end.
new_ws = wb.create_sheet(title="NewDataSheet", index=0)

# 3. Add data to your new sheet
new_ws = df

# 4. Save the file with the macro-enabled extension
wb.save(file_path)

In [ ]:
# #put the sample information into the existing Excel file being setup for Dalhousie in append mode ('a') 
# with pd.ExcelWriter(os.path.join(data_dir,fName), mode='a', engine='openpyxl') as writer:
#     # put it in as a new sheet
#     df.to_excel(writer, sheet_name='projectDetails', index=False)